In [ ]:
# 실험 목표: 데이터 증강(Flip + Crop + ColorJitter)의 유무가 학습 곡선에 미치는 영향을 비교합니다.

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── 1. 변환 파이프라인 ──
no_aug_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
])

aug_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(32, padding=4),
    T.ColorJitter(brightness=0.2, contrast=0.2,
                  saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
])

# ── 2. 간단한 CNN 모델 ──
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                        # 8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                                # 1x1
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# ── 3. 학습 + 평가 함수 ──
def train_and_eval(train_transform, name, epochs=5):
    train_set = CIFAR10('data', train=True,  download=True, transform=train_transform)
    val_set   = CIFAR10('data', train=False, download=True, transform=no_aug_transform)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = SimpleCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    print(f"\n[{name}] 학습 시작 ({epochs} epochs)")
    for epoch in range(1, epochs + 1):
        # — Train —
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # — Validate —
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
        val_acc = 100.0 * correct / total

        print(f"  Epoch {epoch}/{epochs} | train_loss={train_loss:.3f} | val_acc={val_acc:.1f}%")

    return val_acc

# ── 4. 두 조건 비교 실행 ──
acc_no_aug = train_and_eval(no_aug_transform, "No Augmentation")
acc_aug    = train_and_eval(aug_transform,    "With Augmentation")

print("\n" + "="*50)
print(f"결과 요약 (5 epochs, CIFAR-10)")
print(f"  증강 없음  : val_acc = {acc_no_aug:.1f}%")
print(f"  증강 있음  : val_acc = {acc_aug:.1f}%")
print(f"  개선폭     : {acc_aug - acc_no_aug:+.1f}%p")
print("="*50)


# 관찰:
# - 5 에폭의 짧은 학습에서도 증강이 없으면 train loss가 빠르게 내려가지만 val_acc 성장이 정체됩니다.
# - 증강 적용 시 train loss는 더 천천히 내려가지만 val_acc는 더 높게 유지됩니다.

# 해석:
# - **과적합 갭**: 증강 없이 학습하면 모델이 훈련 이미지의 특정 방향·색상에 과적합합니다. 증강은 동일 이미지의 변형 버전을 지속적으로 제공해 "사실상 더 큰 데이터셋"처럼 작동합니다.
# - **Flip+Crop의 역할**: RandomCrop(padding=4)은 이동 불변성을, RandomHorizontalFlip은 좌우 불변성을 학습시킵니다. ColorJitter는 조명 변화에 강건한 표현을 만듭니다.
# - **실전 팁**: 증강 없는 베이스라인을 먼저 측정하고 증강을 하나씩 추가하면 각 기법의 기여분을 정확히 파악할 수 있습니다.

In [ ]:
# 실험 목표: 세 가지 고급 증강 기법(Cutout, Mixup, CutMix)의 CIFAR-10 테스트 정확도를 비교합니다.

import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── 테스트용 변환 (증강 없음, 고정) ──
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
])

# ── p 값에 따라 증강 강도를 조절하는 파이프라인 ──
def make_transform(p: float):
    """
    p=0.0 : 증강 없음 (베이스라인)
    p=0.5 : 표준 증강 (권장값)
    p=1.0 : 모든 증강을 항상 적용 (과도한 증강)
    """
    transforms = [T.RandomHorizontalFlip(p=p)]

    if p > 0.0:
        transforms.append(T.RandomCrop(32, padding=4))   # 항상 crop (p가 클수록 의미 있음)

    rotation_deg = int(30 * p)
    if rotation_deg > 0:
        transforms.append(T.RandomRotation(rotation_deg))

    transforms.append(T.ColorJitter(
        brightness=0.4 * p,
        contrast=0.4 * p,
        saturation=0.4 * p,
        hue=min(0.1 * p, 0.5),   # hue는 최대 0.5
    ))

    if p >= 0.2:
        transforms.append(T.RandomGrayscale(p=p * 0.2))

    transforms += [
        T.ToTensor(),
        T.Normalize([0.4914, 0.4822, 0.4465],
                    [0.2470, 0.2435, 0.2616]),
    ]
    return T.Compose(transforms)

# ── SimpleCNN ──
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# ── 학습 + 평가 함수 ──
def train_eval(p_value, epochs=20):
    torch.manual_seed(42)                         # 각 실험 동일 시드
    train_transform = make_transform(p_value)
    train_set = CIFAR10('data', train=True,  download=True, transform=train_transform)
    val_set   = CIFAR10('data', train=False, download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = SimpleCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(images), labels).backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (model(images).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

# ── p 값 스윕 실행 ──
p_values = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
results  = {}

for p in p_values:
    print(f"p={p:.1f} 학습 중...", flush=True)
    acc = train_eval(p)
    results[p] = acc
    print(f"  → val_acc = {acc:.1f}%")

# ── 결과 요약 ──
best_p   = max(results, key=results.get)
best_acc = results[best_p]

print("\n" + "="*45)
print(f"{'p 값':<10} {'Val Acc':>10} {'비고'}")
print("-"*45)
for p, acc in results.items():
    star = " ★ Best" if p == best_p else ""
    print(f"  p={p:.1f}    {acc:>9.1f}%{star}")
print("="*45)
print(f"\n최적 p = {best_p:.1f} (val_acc={best_acc:.1f}%)")


# 관찰:
# - p=0.0(증강 없음)과 p=0.9(과도한 증강) 모두 중간 p 값보다 낮은 정확도를 보입니다.
# - p=0.4~0.6 구간에서 최고 성능이 나타나는 역U자형 곡선이 관찰됩니다.

# 해석:
# - **너무 약한 증강(p≈0)**: 모델이 훈련 이미지에 과적합합니다. 테스트 이미지의 사소한 변화에도 분류 실패가 발생합니다.
# - **너무 강한 증강(p≈0.9)**: 모든 이미지가 항상 심하게 변형되어 원본 분포와 멀어집니다. 모델이 "정상 이미지"를 충분히 보지 못해 오히려 성능이 떨어집니다.
# - **p=0.5가 표준인 이유**: RandomHorizontalFlip(p=0.5)가 관례적 기본값인 것은 이 역U자형 관계를 경험적으로 확인한 결과입니다.
# - **실전 팁**: 데이터셋이 작을수록 최적 p가 높아지고, 클수록 낮아집니다. AutoAugment·RandAugment는 이 최적 p와 증강 조합을 자동 탐색합니다.

Using device: cuda


c:\Users\lmhst\git\roboticsCV\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



[Baseline] 학습 시작
  Epoch 1/5 | val_acc=48.1%
  Epoch 2/5 | val_acc=56.2%
  Epoch 3/5 | val_acc=59.2%
  Epoch 4/5 | val_acc=62.8%
  Epoch 5/5 | val_acc=62.5%

[Cutout] 학습 시작


: 

In [3]:
# 실험 목표: 증강 적용 확률(p)을 변경하면서 테스트 정확도의 변화를 관찰합니다. "적절한 강도"가 존재하는지 확인합니다.


import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── 테스트용 변환 (증강 없음, 고정) ──
test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
])

# ── p 값에 따라 증강 강도를 조절하는 파이프라인 ──
def make_transform(p: float):
    """
    p=0.0 : 증강 없음 (베이스라인)
    p=0.5 : 표준 증강 (권장값)
    p=1.0 : 모든 증강을 항상 적용 (과도한 증강)
    """
    transforms = [T.RandomHorizontalFlip(p=p)]

    if p > 0.0:
        transforms.append(T.RandomCrop(32, padding=4))   # 항상 crop (p가 클수록 의미 있음)

    rotation_deg = int(30 * p)
    if rotation_deg > 0:
        transforms.append(T.RandomRotation(rotation_deg))

    transforms.append(T.ColorJitter(
        brightness=0.4 * p,
        contrast=0.4 * p,
        saturation=0.4 * p,
        hue=min(0.1 * p, 0.5),   # hue는 최대 0.5
    ))

    if p >= 0.2:
        transforms.append(T.RandomGrayscale(p=p * 0.2))

    transforms += [
        T.ToTensor(),
        T.Normalize([0.4914, 0.4822, 0.4465],
                    [0.2470, 0.2435, 0.2616]),
    ]
    return T.Compose(transforms)

# ── SimpleCNN ──
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# ── 학습 + 평가 함수 ──
def train_eval(p_value, epochs=10):
    torch.manual_seed(42)                         # 각 실험 동일 시드
    train_transform = make_transform(p_value)
    train_set = CIFAR10('data', train=True,  download=True, transform=train_transform)
    val_set   = CIFAR10('data', train=False, download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False,
                              num_workers=2, pin_memory=True)

    model     = SimpleCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(images), labels).backward()
            optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (model(images).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

# ── p 값 스윕 실행 ──
p_values = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
results  = {}

for p in p_values:
    print(f"p={p:.1f} 학습 중...", flush=True)
    acc = train_eval(p)
    results[p] = acc
    print(f"  → val_acc = {acc:.1f}%")

# ── 결과 요약 ──
best_p   = max(results, key=results.get)
best_acc = results[best_p]

print("\n" + "="*45)
print(f"{'p 값':<10} {'Val Acc':>10} {'비고'}")
print("-"*45)
for p, acc in results.items():
    star = " ★ Best" if p == best_p else ""
    print(f"  p={p:.1f}    {acc:>9.1f}%{star}")
print("="*45)
print(f"\n최적 p = {best_p:.1f} (val_acc={best_acc:.1f}%)")


# 관찰:
# - p=0.0(증강 없음)과 p=0.9(과도한 증강) 모두 중간 p 값보다 낮은 정확도를 보입니다.
# - p=0.4~0.6 구간에서 최고 성능이 나타나는 역U자형 곡선이 관찰됩니다.

# 해석:
# - **너무 약한 증강(p≈0)**: 모델이 훈련 이미지에 과적합합니다. 테스트 이미지의 사소한 변화에도 분류 실패가 발생합니다.
# - **너무 강한 증강(p≈0.9)**: 모든 이미지가 항상 심하게 변형되어 원본 분포와 멀어집니다. 모델이 "정상 이미지"를 충분히 보지 못해 오히려 성능이 떨어집니다.
# - **p=0.5가 표준인 이유**: RandomHorizontalFlip(p=0.5)가 관례적 기본값인 것은 이 역U자형 관계를 경험적으로 확인한 결과입니다.
# - **실전 팁**: 데이터셋이 작을수록 최적 p가 높아지고, 클수록 낮아집니다. AutoAugment·RandAugment는 이 최적 p와 증강 조합을 자동 탐색합니다.

Using device: cuda
p=0.0 학습 중...
  → val_acc = 67.2%
p=0.1 학습 중...
  → val_acc = 67.9%
p=0.3 학습 중...
  → val_acc = 69.2%
p=0.5 학습 중...
  → val_acc = 67.7%
p=0.7 학습 중...
  → val_acc = 66.2%
p=0.9 학습 중...
  → val_acc = 63.1%

p 값           Val Acc 비고
---------------------------------------------
  p=0.0         67.2%
  p=0.1         67.9%
  p=0.3         69.2% ★ Best
  p=0.5         67.7%
  p=0.7         66.2%
  p=0.9         63.1%

최적 p = 0.3 (val_acc=69.2%)


In [ ]:
# ?? ??: TTA ?? ??? view ?? ?? ???/???? ??? ?????.

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from PIL import Image

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

# 1) ?? ??
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# 2) ?? ??
train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(32, padding=4),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
standard_test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

print("?? ?? ? (10 epochs)...")
train_set = CIFAR10('data', train=True, download=True, transform=train_transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

model = SimpleCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 11):
    model.train()
    epoch_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"  Epoch {epoch}/10 | loss={epoch_loss/len(train_loader):.3f}")

# 3) ?? ?? ??? ??
def eval_standard(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (model(images).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

val_set_std = CIFAR10('data', train=False, download=True, transform=standard_test_transform)
val_loader = DataLoader(val_set_std, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
acc_standard = eval_standard(model, val_loader)
print(f"?? ?? (1 view): {acc_standard:.2f}%")

# 4) TTA ?? ??
def _base(extra=None):
    ops = (extra or []) + [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(ops)

def hflip(img):
    return T.functional.hflip(img)

tta_transforms = [
    _base(),
    _base([T.Lambda(hflip)]),
    _base([T.Pad(2), T.CenterCrop(32)]),
    _base([T.Pad(2), T.FiveCrop(32), T.Lambda(lambda c: c[0])]),
    _base([T.Pad(2), T.FiveCrop(32), T.Lambda(lambda c: c[4])]),
    _base([T.Lambda(hflip), T.Pad(2), T.CenterCrop(32)]),
]

tta_transforms_20 = tta_transforms + [
    _base([T.Lambda(hflip), T.Pad(2), T.FiveCrop(32), T.Lambda(lambda c: c[0])]),
    _base([T.Lambda(hflip), T.Pad(2), T.FiveCrop(32), T.Lambda(lambda c: c[4])]),
    _base([T.Pad(2), T.FiveCrop(32), T.Lambda(lambda c: c[2])]),
    _base([T.Lambda(lambda x: T.functional.rotate(x, 8))]),
    _base([T.Lambda(lambda x: T.functional.rotate(x, -8))]),
    _base([T.ColorJitter(brightness=0.12, contrast=0.12)]),
    _base([T.ColorJitter(saturation=0.12, hue=0.02)]),
    _base([T.Lambda(lambda x: T.functional.hflip(T.functional.rotate(x, 8)))]),
    _base([T.Lambda(lambda x: T.functional.hflip(T.functional.rotate(x, -8)))]),
    _base([T.GaussianBlur(kernel_size=3, sigma=(0.3, 0.3))]),
    _base([T.Lambda(lambda x: T.functional.adjust_sharpness(x, 1.4))]),
    _base([T.Lambda(lambda x: T.functional.adjust_contrast(x, 1.2))]),
    _base([T.RandomAutocontrast(p=1.0)]),
    _base([T.RandomEqualize(p=1.0)]),
]

def eval_tta(model, dataset, transform_list, max_samples=2000):
    model.eval()
    correct = total = 0
    indices = range(min(max_samples, len(dataset)))

    with torch.no_grad():
        for i in indices:
            raw_img_np, label = dataset.data[i], dataset.targets[i]
            pil_img = Image.fromarray(raw_img_np)

            probs_list = []
            for tfm in transform_list:
                tensor = tfm(pil_img).unsqueeze(0).to(DEVICE)
                logits = model(tensor)
                probs_list.append(F.softmax(logits, dim=1))

            avg_probs = torch.stack(probs_list).mean(0)
            pred = avg_probs.argmax(1).item()
            correct += int(pred == label)
            total += 1

    acc = 100.0 * correct / total
    return acc

# raw dataset
val_raw = CIFAR10('data', train=False, download=True, transform=None)

acc_std_subset = eval_tta(model, val_raw,
                           [_base()], max_samples=2000)
acc_tta_2view  = eval_tta(model, val_raw,
                           tta_transforms[:2], max_samples=2000)
acc_tta_6view  = eval_tta(model, val_raw,
                           tta_transforms,     max_samples=2000)
acc_tta_20view = eval_tta(model, val_raw,
                           tta_transforms_20,  max_samples=2000)

print("\n" + "="*55)
print(f"{'Method':<30} {'Val Acc':>10} {'Views':>10}")
print("-"*55)
print(f"  {'Standard (1 view)':<28} {acc_std_subset:>9.2f}%  {'1x':>9}")
print(f"  {'TTA 2-view (orig+flip)':<28} {acc_tta_2view:>9.2f}%  {'2x':>9}")
print(f"  {'TTA 6-view (crops+flips)':<28} {acc_tta_6view:>9.2f}%  {'6x':>9}")
print(f"  {'TTA 20-view':<28} {acc_tta_20view:>9.2f}% {'20x':>10}")
print("="*55)
print(f"2-view TTA gain: {acc_tta_2view - acc_std_subset:+.2f}%p")
print(f"6-view TTA gain: {acc_tta_6view - acc_std_subset:+.2f}%p")
print(f"20-view TTA gain: {acc_tta_20view - acc_std_subset:+.2f}%p")



